# Day 026 | Plotly — Interactive Plots

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `plotly.express` | Quick interactive charts in one line | Scatter plot of N-value vs depth with hover |
| `go.Scatter` | Fine-grained scatter/line control | Custom borehole profile with markers |
| `go.Bar` | Interactive bar charts | SPT blow counts per depth interval |
| Hover tooltips | Info on mouseover | Show soil type, N-value, depth at each point |
| 3D scatter | Three-dimensional visualization | Spatial N-value distribution (x, y, depth) |
| Dropdown menus | Interactive filtering | Select borehole to display |
| Subplots | Multiple panels | Side-by-side borehole comparisons |

---

## Our Geotechnical Problem

A site investigation for a high-rise building has **5 boreholes** across the site. The project manager wants an **interactive dashboard** where they can:

- Hover over data points to see detailed soil properties  
- Select individual boreholes from a dropdown  
- View the 3D spatial distribution of soil strength  
- Zoom, pan, and export plots for reports  

Static plots cannot do this. Plotly creates interactive HTML visualizations that work in Jupyter notebooks and can be shared as standalone files.

---

## Part 1: Generating Site Investigation Data

We create a realistic dataset with 5 boreholes at known coordinates, each drilled to different depths with varying soil conditions.

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATING SITE INVESTIGATION DATA
# ─────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# Borehole locations (x, y coordinates in meters from site origin)
borehole_info = {
    'BH-01': {'x': 10, 'y': 15, 'max_depth': 20},
    'BH-02': {'x': 35, 'y': 12, 'max_depth': 18},
    'BH-03': {'x': 25, 'y': 40, 'max_depth': 22},
    'BH-04': {'x': 50, 'y': 35, 'max_depth': 15},
    'BH-05': {'x': 42, 'y': 55, 'max_depth': 20}
}

rows = []
soil_sequence = {
    'BH-01': ['Fill', 'Clay', 'Clay', 'Sand', 'Sand', 'Gravel', 'Gravel', 'Rock'],
    'BH-02': ['Fill', 'Sand', 'Clay', 'Clay', 'Sand', 'Gravel', 'Rock', 'Rock'],
    'BH-03': ['Fill', 'Clay', 'Silt', 'Clay', 'Sand', 'Sand', 'Gravel', 'Gravel', 'Rock'],
    'BH-04': ['Fill', 'Sand', 'Sand', 'Gravel', 'Gravel', 'Rock'],
    'BH-05': ['Fill', 'Silt', 'Clay', 'Sand', 'Sand', 'Gravel', 'Gravel', 'Rock']
}

for bh_id, info in borehole_info.items():
    depths = np.arange(1.5, info['max_depth'] + 0.1, 1.5)
    soils = soil_sequence[bh_id]
    for i, d in enumerate(depths):
        soil_idx = min(i, len(soils) - 1)
        soil = soils[soil_idx]
        # Generate N-values that increase with depth and vary by soil
        base_n = {'Fill': 4, 'Clay': 6, 'Silt': 8, 'Sand': 12, 'Gravel': 25, 'Rock': 50}[soil]
        n_val = max(2, int(base_n + 1.5 * d + np.random.normal(0, 3)))
        wc = max(5, round({'Fill': 20, 'Clay': 32, 'Silt': 25, 'Sand': 15, 'Gravel': 8, 'Rock': 3}[soil]
                          - 0.3 * d + np.random.normal(0, 3), 1))
        gamma = round({'Fill': 16.5, 'Clay': 17.5, 'Silt': 18.0, 'Sand': 18.5, 'Gravel': 20.0, 'Rock': 22.0}[soil]
                      + 0.05 * d + np.random.normal(0, 0.3), 1)
        rows.append({
            'borehole_id': bh_id, 'x_coord': info['x'], 'y_coord': info['y'],
            'depth_m': round(d, 1), 'N_value': n_val, 'soil_type': soil,
            'water_content': wc, 'unit_weight': gamma
        })

df = pd.DataFrame(rows)
print(f"Dataset: {len(df)} records from {df['borehole_id'].nunique()} boreholes")
print(f"Depth range: {df['depth_m'].min()} - {df['depth_m'].max()} m")
print(f"Soil types: {df['soil_type'].unique().tolist()}")
print()
df.head(10)

---

## Part 2: Plotly Express — Quick Interactive Scatter

`plotly.express` creates interactive plots in a single function call. Every data point can be hovered to reveal details — no more squinting at static plots.

In [ ]:
# ─────────────────────────────────────────────────────
# PLOTLY EXPRESS — Interactive scatter plot
# ─────────────────────────────────────────────────────

fig = px.scatter(
    df, x='N_value', y='depth_m',
    color='soil_type',
    symbol='borehole_id',
    size='unit_weight',
    hover_data=['water_content', 'unit_weight', 'borehole_id'],
    title='SPT N-value vs Depth — All Boreholes',
    labels={'N_value': 'SPT N-value (blows/30cm)',
            'depth_m': 'Depth (m)',
            'soil_type': 'Soil Type'}
)

# Invert y-axis (depth increases downward)
fig.update_yaxes(autorange='reversed')
fig.update_layout(height=550, width=800)

fig.show()

print("Hover over any point to see its soil type, N-value, depth, and more.")
print("Click legend items to toggle soil types on/off.")
print("Use the toolbar to zoom, pan, or download as PNG.")

---

## Part 3: `go.Bar` — Interactive Bar Chart

`go.Bar` creates interactive bar charts. We will plot SPT blow counts as horizontal bars to create a simplified borehole log — a visualization every geotechnical engineer recognizes.

In [ ]:
# ─────────────────────────────────────────────────────
# go.Bar — SPT Borehole Log
# ─────────────────────────────────────────────────────

bh01 = df[df['borehole_id'] == 'BH-01'].sort_values('depth_m')

# Color bars by soil type
soil_colors = {'Fill': '#D4A574', 'Clay': '#8B7355', 'Silt': '#C4B896',
               'Sand': '#F5DEB3', 'Gravel': '#A9A9A9', 'Rock': '#696969'}
colors = [soil_colors[s] for s in bh01['soil_type']]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=[f"{d:.1f}m" for d in bh01['depth_m']],
    x=bh01['N_value'],
    orientation='h',
    marker_color=colors,
    text=[f"{s} (N={n})" for s, n in zip(bh01['soil_type'], bh01['N_value'])],
    textposition='outside',
    hovertemplate='Depth: %{y}<br>N-value: %{x}<br>%{text}<extra></extra>'
))

fig.update_layout(
    title='BH-01 — SPT Borehole Log',
    xaxis_title='SPT N-value (blows/30cm)',
    yaxis_title='Depth',
    yaxis=dict(autorange='reversed'),
    height=500, width=650,
    showlegend=False
)

# Add reference lines for consistency classification
for n_ref, label in [(4, 'Very Loose'), (10, 'Loose'), (30, 'Dense'), (50, 'V.Dense')]:
    fig.add_vline(x=n_ref, line_dash='dot', line_color='gray', opacity=0.5,
                  annotation_text=label, annotation_position='top')

fig.show()

print("Horizontal bars show N-value at each depth — the classic SPT log format.")
print("Dotted lines mark density classification boundaries.")

---

## Part 4: Interactive Borehole Profile with `go.Scatter`

`go.Scatter` gives full control over markers, lines, and hover text. We create a detailed borehole profile showing N-values with soil type coloring and rich hover tooltips.

In [ ]:
# ─────────────────────────────────────────────────────
# go.Scatter — Detailed borehole profile with hover
# ─────────────────────────────────────────────────────

fig = go.Figure()

for bh_id in ['BH-01', 'BH-02', 'BH-03']:
    bh = df[df['borehole_id'] == bh_id].sort_values('depth_m')
    
    hover_text = [
        f"<b>{bh_id}</b><br>"
        f"Depth: {d:.1f} m<br>"
        f"N-value: {n}<br>"
        f"Soil: {s}<br>"
        f"w = {w}%<br>"
        f"γ = {g} kN/m³"
        for d, n, s, w, g in zip(bh['depth_m'], bh['N_value'],
                                  bh['soil_type'], bh['water_content'],
                                  bh['unit_weight'])
    ]
    
    fig.add_trace(go.Scatter(
        x=bh['N_value'], y=bh['depth_m'],
        mode='lines+markers',
        name=bh_id,
        marker=dict(size=10, line=dict(width=1, color='white')),
        line=dict(width=2),
        hovertext=hover_text,
        hoverinfo='text'
    ))

fig.update_layout(
    title='Borehole Profiles — N-value vs Depth',
    xaxis_title='SPT N-value (blows/30cm)',
    yaxis_title='Depth (m)',
    yaxis=dict(autorange='reversed'),
    height=550, width=750,
    hovermode='closest'
)

fig.show()

print("Hover over any point to see complete soil properties.")
print("Click a borehole in the legend to hide/show it.")
print("Double-click a legend item to isolate just that borehole.")

---

## Part 5: 3D Scatter — Spatial Distribution

Plotly's 3D scatter plots let you visualize how soil properties vary in three dimensions: borehole X-coordinate, Y-coordinate, and depth. The engineer can rotate and explore the subsurface model interactively.

In [ ]:
# ─────────────────────────────────────────────────────
# 3D SCATTER — Spatial N-value distribution
# ─────────────────────────────────────────────────────

fig = px.scatter_3d(
    df,
    x='x_coord', y='y_coord', z='depth_m',
    color='N_value',
    symbol='borehole_id',
    size='N_value',
    size_max=15,
    color_continuous_scale='Viridis',
    hover_data=['soil_type', 'water_content', 'unit_weight'],
    title='3D Subsurface Model — SPT N-value Distribution',
    labels={'x_coord': 'Easting (m)', 'y_coord': 'Northing (m)',
            'depth_m': 'Depth (m)', 'N_value': 'N-value'}
)

# Invert z-axis so depth increases downward
fig.update_scenes(zaxis=dict(autorange='reversed'))
fig.update_layout(height=600, width=800)

fig.show()

print("Click and drag to rotate the 3D model.")
print("Scroll to zoom. Hover for soil details at each point.")
print("Larger, brighter dots = higher N-values (stronger soil).")

---

## Part 6: Dropdown Menu for Borehole Selection

Plotly supports interactive controls like dropdown menus. We create a plot where the user can select which borehole to display — perfect for project manager presentations.

In [ ]:
# ─────────────────────────────────────────────────────
# DROPDOWN MENU — Select borehole to display
# ─────────────────────────────────────────────────────

fig = go.Figure()

borehole_ids = df['borehole_id'].unique().tolist()

# Add one trace per borehole
for bh_id in borehole_ids:
    bh = df[df['borehole_id'] == bh_id].sort_values('depth_m')
    fig.add_trace(go.Scatter(
        x=bh['N_value'], y=bh['depth_m'],
        mode='lines+markers',
        name=bh_id,
        marker=dict(size=10),
        line=dict(width=2),
        hovertemplate=(
            f'<b>{bh_id}</b><br>'
            'Depth: %{y:.1f} m<br>'
            'N-value: %{x}<br>'
            '<extra></extra>'
        ),
        visible=(bh_id == borehole_ids[0])  # only first visible initially
    ))

# Create dropdown buttons
buttons = []

# "All Boreholes" button
buttons.append(dict(
    label='All Boreholes',
    method='update',
    args=[{'visible': [True] * len(borehole_ids)}]
))

# Individual borehole buttons
for i, bh_id in enumerate(borehole_ids):
    visibility = [False] * len(borehole_ids)
    visibility[i] = True
    buttons.append(dict(
        label=bh_id,
        method='update',
        args=[{'visible': visibility}]
    ))

fig.update_layout(
    title='Select Borehole to View',
    xaxis_title='SPT N-value',
    yaxis_title='Depth (m)',
    yaxis=dict(autorange='reversed'),
    height=550, width=700,
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        x=0.98, y=1.15,
        showactive=True,
        buttons=buttons
    )]
)

fig.show()

print("Use the dropdown to select individual boreholes or view all.")
print("This is how you deliver interactive reports to project managers.")

---

## Part 7: Plotly Express Extras — Box and Strip Plots

Plotly Express also provides statistical plot types with built-in interactivity.

In [ ]:
# ─────────────────────────────────────────────────────
# PLOTLY EXPRESS — Box and strip plots
# ─────────────────────────────────────────────────────

# Interactive box plot — N-value by soil type
fig = px.box(
    df, x='soil_type', y='N_value', color='soil_type',
    points='all',  # show individual data points
    title='SPT N-value Distribution by Soil Type',
    labels={'N_value': 'N-value (blows/30cm)', 'soil_type': 'Soil Type'},
    category_orders={'soil_type': ['Fill', 'Clay', 'Silt', 'Sand', 'Gravel', 'Rock']}
)

fig.update_layout(height=450, width=700, showlegend=False)
fig.show()

print("Hover over the box to see median, Q1, Q3, and whisker values.")
print("Individual points show the actual data distribution.")
print("Rock and Gravel have the highest N-values, as expected.")

---

## Part 8: Complete Application — Interactive Site Investigation Dashboard

Let us combine all Plotly techniques into a comprehensive interactive dashboard with multiple panels, showing the complete site investigation in one figure.

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE: INTERACTIVE SITE INVESTIGATION DASHBOARD
# ══════════════════════════════════════════════════════

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Borehole Profiles — N-value vs Depth',
        'Site Plan — Borehole Locations',
        'N-value by Soil Type',
        'Water Content vs Depth'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'scatter'}]
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

colors = {'BH-01': '#1f77b4', 'BH-02': '#ff7f0e', 'BH-03': '#2ca02c',
           'BH-04': '#d62728', 'BH-05': '#9467bd'}

# ── Panel 1: N-value profiles ──
for bh_id in borehole_ids:
    bh = df[df['borehole_id'] == bh_id].sort_values('depth_m')
    fig.add_trace(go.Scatter(
        x=bh['N_value'], y=bh['depth_m'],
        mode='lines+markers', name=bh_id,
        marker=dict(size=7, color=colors[bh_id]),
        line=dict(width=2, color=colors[bh_id]),
        hovertemplate=f'<b>{bh_id}</b><br>N=%{{x}}, Depth=%{{y:.1f}}m<extra></extra>',
        legendgroup=bh_id
    ), row=1, col=1)

# ── Panel 2: Site plan (borehole locations) ──
for bh_id in borehole_ids:
    info = borehole_info[bh_id]
    avg_n = df[df['borehole_id'] == bh_id]['N_value'].mean()
    fig.add_trace(go.Scatter(
        x=[info['x']], y=[info['y']],
        mode='markers+text',
        marker=dict(size=20, color=colors[bh_id],
                    line=dict(width=2, color='white')),
        text=[bh_id], textposition='top center',
        hovertemplate=(f'<b>{bh_id}</b><br>'
                       f'Location: ({info["x"]}, {info["y"]}) m<br>'
                       f'Max depth: {info["max_depth"]} m<br>'
                       f'Avg N: {avg_n:.0f}<extra></extra>'),
        showlegend=False
    ), row=1, col=2)

# ── Panel 3: N-value by soil type ──
soil_order = ['Fill', 'Clay', 'Silt', 'Sand', 'Gravel', 'Rock']
soil_colors_map = {'Fill': '#D4A574', 'Clay': '#8B7355', 'Silt': '#C4B896',
                   'Sand': '#F5DEB3', 'Gravel': '#A9A9A9', 'Rock': '#696969'}

for soil in soil_order:
    soil_data = df[df['soil_type'] == soil]
    if len(soil_data) > 0:
        fig.add_trace(go.Box(
            y=soil_data['N_value'], name=soil,
            marker_color=soil_colors_map[soil],
            showlegend=False,
            hoverinfo='y+name'
        ), row=2, col=1)

# ── Panel 4: Water content vs depth ──
for bh_id in borehole_ids:
    bh = df[df['borehole_id'] == bh_id].sort_values('depth_m')
    fig.add_trace(go.Scatter(
        x=bh['water_content'], y=bh['depth_m'],
        mode='markers', name=bh_id,
        marker=dict(size=8, color=colors[bh_id]),
        hovertemplate=f'<b>{bh_id}</b><br>w=%{{x}}%, Depth=%{{y:.1f}}m<extra></extra>',
        showlegend=False, legendgroup=bh_id
    ), row=2, col=2)

# Update axes
fig.update_yaxes(autorange='reversed', title_text='Depth (m)', row=1, col=1)
fig.update_xaxes(title_text='N-value', row=1, col=1)
fig.update_xaxes(title_text='Easting (m)', row=1, col=2)
fig.update_yaxes(title_text='Northing (m)', row=1, col=2)
fig.update_yaxes(title_text='N-value', row=2, col=1)
fig.update_xaxes(title_text='Water Content (%)', row=2, col=2)
fig.update_yaxes(autorange='reversed', title_text='Depth (m)', row=2, col=2)

fig.update_layout(
    title_text='SITE INVESTIGATION DASHBOARD — Interactive Overview',
    title_font_size=16,
    height=800, width=950,
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)

fig.show()

print("This dashboard gives a complete interactive overview of the site.")
print("All four panels support hover, zoom, and pan independently.")
print("Click borehole names in the legend to toggle visibility across panels.")

---

## Practice Exercises

### Exercise 1: Animated Scatter
Use `px.scatter()` with `animation_frame='depth_m'` to create an animated plot showing how N-values change with depth across all boreholes.

### Exercise 2: Heatmap of N-values
Create a `go.Heatmap` showing mean N-value for each borehole (columns) at each depth interval (rows). This is a 2D fence diagram.

### Exercise 3: Export to HTML
Save any of the dashboard plots as a standalone HTML file using `fig.write_html('dashboard.html')`. Open it in a browser to verify the interactivity works without Jupyter.

In [ ]:
# EXERCISE 1
# fig = px.scatter(df, x='x_coord', y='y_coord', color='N_value',
#                  size='N_value', animation_frame='depth_m',
#                  range_color=[0, 60], size_max=20,
#                  title='N-value by Depth (animated)')
# fig.show()

In [ ]:
# EXERCISE 2
# pivot = df.pivot_table(values='N_value', index='depth_m',
#                        columns='borehole_id', aggfunc='mean')
# fig = go.Figure(data=go.Heatmap(
#     z=pivot.values, x=pivot.columns, y=pivot.index,
#     colorscale='Viridis', colorbar_title='N-value'))
# fig.update_yaxes(autorange='reversed')
# fig.update_layout(title='N-value Heatmap', yaxis_title='Depth (m)')
# fig.show()

In [ ]:
# EXERCISE 3
# fig.write_html('site_dashboard.html')
# print('Dashboard saved as site_dashboard.html')
# print('Open in any browser for full interactivity.')

---

## Key Takeaways

1. **`plotly.express`** creates interactive plots in one line. Hover, zoom, pan, and legend toggling work automatically. Perfect for quick exploration.

2. **`go.Scatter`** and **`go.Bar`** give full control over markers, lines, and hover templates. Use them for custom borehole logs with rich tooltips showing all soil properties.

3. **3D scatter plots** let you visualize subsurface conditions spatially — N-value at (x, y, depth). The engineer can rotate and explore the model interactively.

4. **Dropdown menus** make your plots presentation-ready. Project managers can select boreholes without touching code.

5. **`fig.write_html()`** exports any Plotly figure as a standalone interactive HTML file — no Python needed to view it. Share directly with clients.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Interactive scatter plots | Exploring feature spaces before modeling |
| Hover tooltips | Inspecting individual predictions and residuals |
| 3D scatter | Visualizing high-dimensional embeddings (PCA, t-SNE) |
| Dropdown filtering | Model comparison dashboards (select algorithm) |
| `make_subplots` | Training curves, confusion matrices, feature importance in one view |
| HTML export | Sharing ML results with non-technical stakeholders |
| Color-coded markers | Cluster visualization from k-means or DBSCAN |
| Animation frames | Showing how model predictions change with hyperparameters |

Plotly bridges the gap between data science and engineering practice. Your ML model's predictions need to be communicated to engineers and project managers — and interactive plots are far more effective than static images.

---

*Day 026 of 100 — Interactive plots turn data exploration into a conversation!*